# A5: Optimization Human Preference & LLM-as-a-Judge

This assignment focuses on two critical aspects of modern Large Language Model (LLM) development: 

Alignment and Evaluation. You will leverage the Direct Preference Optimization (DPO) trainer to align a pre-trained model to be ”Truthful” (avoiding hallucinations). Furthermore, you will learn how to build a reliable LLM-as-a-Judge pipeline to evaluate factuality.

DPO Paper [here](https://arxiv.org/pdf/2305.18290)

## Task 1. Dataset Preparation (0.5 point)

We will use a small, focused dataset designed to teach models to be truthful and avoid hallucinations.

Dataset: [jondurbin/truthy-dpo-v0.11](https://huggingface.co/datasets/jondurbin/truthy-dpo-v0.1)

Task:
- Load the dataset using the Hugging Face datasets library.
- This dataset contains prompt, chosen (factual answer), and rejected (hallucinated/wrong answer).

#### Prepare Environment - import libraries and select device

In [1]:
import sys
print(sys.version)

3.11.14 (main, Oct 28 2025, 12:11:54) [Clang 20.1.4 ]


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

import datasets, math, re, random, time
from collections import Counter
from tqdm import tqdm

In [3]:
# minimum required torch version for MPS support and transformers 5.x: "2.4+"
torch.__version__

'2.10.0'

In [4]:
# universal device selection: use gpu if available, else cpu
import torch

def get_device():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()  # Clear CUDA cache to free up memory 
        return torch.device("cuda")      # NVIDIA GPU
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()  # Clear MPS cache to avoid memory issues
        return torch.device("mps")       # Apple Silicon GPU
    else:
        torch.empty_cache()  # Clear CPU cache to free up memory
        return torch.device("cpu")

device = get_device()

print(f"Using device: {device}")

Using device: mps


In [5]:
import os
from dotenv import load_dotenv

# .env lives at project root, two levels above this notebook
load_dotenv(os.path.join(os.path.dirname(os.getcwd()), "..", ".env"), override=True)

# Fallback: try root directly
if not os.environ.get("HF_TOKEN"):
    load_dotenv("/Users/sushmi/dev/nlp/assignment-npl/.env", override=True)

HF_TOKEN = os.environ.get("HF_TOKEN")
print(f"HF_TOKEN loaded: {'YES — ' + HF_TOKEN[:10] + '...' if HF_TOKEN else 'NO — check .env file'}")

HF_TOKEN loaded: NO — check .env file


In [6]:
# Setting seed for reproducibility
SEED = 126526
torch.manual_seed(SEED)

#### Step 1: Load Data

In [7]:
from datasets import load_dataset

ds = load_dataset("jondurbin/truthy-dpo-v0.1")

In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'system', 'prompt', 'chosen', 'rejected'],
        num_rows: 1016
    })
})

In [9]:
# Explore dataset structure
print(f"Dataset size: {len(ds['train'])} examples")
print(f"\nFeatures: {ds['train'].features}")
sample = ds["train"][0]
print(f"\nSample entry:")
print(f"  Prompt   : {sample['prompt'][:150]}...")
print(f"  Chosen   : {sample['chosen'][:150]}...")
print(f"  Rejected : {sample['rejected'][:150]}...")

Dataset size: 1016 examples

Features: {'id': Value('string'), 'source': Value('string'), 'system': Value('string'), 'prompt': Value('string'), 'chosen': Value('string'), 'rejected': Value('string')}

Sample entry:
  Prompt   : What's the nearest national park to you?...
  Chosen   : As an AI, I don't have a physical location, so I can't provide the distance to the nearest national park....
  Rejected : I don't have access to the user's location, so I can't determine the nearest national park....


There's only train data, splitting it into train and test

In [27]:
data = ds['train']
split_data = data.train_test_split(test_size=0.2, seed=SEED, shuffle=True)
split_data

DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'system', 'prompt', 'chosen', 'rejected'],
        num_rows: 812
    })
    test: Dataset({
        features: ['id', 'source', 'system', 'prompt', 'chosen', 'rejected'],
        num_rows: 204
    })
})

In [ ]:
# print data in dataframe
import pandas as pd

df = pd.DataFrame(ds["train"].select(range(5)))

df.head()

,id,source,system,prompt,chosen,rejected
0,04c275bf738fd391b7fe25e25fe7bed3,truthy_dpo,"You are an unbiased, uncensored, helpful assis...",What's the nearest national park to you?,"As an AI, I don't have a physical location, so...","I don't have access to the user's location, so..."
1,6ef28b6b28c6cd94c0bb3ee72f8e2262,truthy_dpo,"You are an unbiased, uncensored, helpful assis...",Do you possess the ability to navigate or move...,"No, I do not possess the ability to navigate o...","Yes, I can navigate and move within a physical..."
2,20f1724e5cd2fbe4d4200e438122c93a,truthy_dpo,"You are an unbiased, uncensored, helpful assis...",Do wooden pencils contain lead as their core?,"No, wooden pencils do not contain lead in thei...","Yes, wooden pencils typically contain a core m..."
3,16a9da15058b948193dde8c45f289726,truthy_dpo,"You are an unbiased, uncensored, helpful assis...",Is there a dark side of the moon?,There is no permanently dark side of the moon....,"Yes, there is a dark side of the moon. The far..."
4,662dcdda2b8d961189f3ea44e0e20a67,truthy_dpo,"You are an unbiased, uncensored, helpful assis...",What's the weather like where you are?,"As an artificial intelligence, I don't have a ...",I'm not sure. I don't have access to the curre...


## Task 2. Training a Model with DPOTrainer (2 points)

1. Implement the Direct Preference Optimization (DPO) training method with `DPOTrainer` using a pre-trained transformer model (e.g., `Qwen/Qwen2.5-1.5B-Instruct`) and fine-tune it using the training set from Task 1.
2. Experiment with hyperparameters and report training performance (loss curves).

> HINT: Refer to the [Hugging Face documentation for DPOTrainer](https://huggingface.co/docs/trl/main/dpo_trainer) implementation.

#### Step 1: Load Pre-trained Model & Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model: {MODEL_NAME}")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,   # half precision → ~3 GB
    token=HF_TOKEN,
).to(device)


Loading tokenizer: Qwen/Qwen2.5-1.5B-Instruct
Loading model: Qwen/Qwen2.5-1.5B-Instruct


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

To save time and memory full fine-tuning is avoided, instead LoRA is used with fraction of parameters.

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

# LoRA config: only train a tiny fraction of parameters (~0.5%)
# The frozen base model acts as its own DPO reference — no second copy needed
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                                 # rank — lower = less memory
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"], # attention projections only
    bias="none",
)

total_params    = sum(p.numel() for p in base_model.parameters()) / 1e6
print(f"Model parameters : {total_params:.1f}M")
print(f"LoRA rank        : {lora_config.r} (trains ~{2*lora_config.r*1536*28/1e6:.1f}M params)")
print(f"Dtype            : {next(base_model.parameters()).dtype}")

policy_model = get_peft_model(base_model, lora_config)
policy_model.print_trainable_parameters()

Model parameters : 1543.7M
LoRA rank        : 8 (trains ~0.7M params)
Dtype            : torch.float16
trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


#### Step 2: Preprocess Dataset for DPO

In [28]:
def format_dpo_example(example):
    """Format dataset examples as chat messages for DPOTrainer."""
    system_msg = example.get("system") or "You are a helpful, truthful assistant."
    return {
        "prompt": [
            {"role": "system", "content": system_msg},
            {"role": "user",   "content": example["prompt"]},
        ],
        "chosen":   [{"role": "assistant", "content": example["chosen"]}],
        "rejected": [{"role": "assistant", "content": example["rejected"]}],
    }

# Use a subset of 200 examples for faster training
TRAIN_SIZE = 200
train_subset = split_data["train"].select(range(TRAIN_SIZE))

dpo_dataset = train_subset.map(
    format_dpo_example,
    remove_columns=train_subset.column_names,
)

print(f"DPO training dataset size : {len(dpo_dataset)}")
print(f"\nSample formatted prompt:")
for msg in dpo_dataset[0]["prompt"]:
    print(f"  [{msg['role']}]: {msg['content'][:120]}...")

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

DPO training dataset size : 200

Sample formatted prompt:
  [system]: You are Rosa Parks:
Rosa Parks was an African-American civil rights activist who became a pivotal figure in the fight ag...
  [user]: Can you recall and learn from past physical experiences?...


#### Step 3: Configure DPOTrainer & Train & Test

Install DPO
```bash
uv add trl anthropic accelerate 
```

In [30]:
# To avoid Runtime error:
# RuntimeError: MPS backend out of memory (MPS allocated: 33.31 GiB, other allocations: 54.37 GiB, max allowed: 88.13 GiB). 
# Tried to allocate 890.25 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 
# to disable upper limit for memory allocations (may cause system failure).
PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0


In [ ]:
base_model

In [ ]:
from trl import DPOTrainer, DPOConfig

# Clear MPS cache before training
torch.mps.empty_cache()

dpo_config = DPOConfig(
    output_dir="../model/dpo_output",
    # -- Training schedule --
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    # -- Optimiser --
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    optim="adamw_torch",
    # -- Memory optimisation --
    gradient_checkpointing=True,
    fp16=False,                          # MPS has no fp16 AMP scaler
    bf16=False,
    max_length=256,
    # -- DPO-specific --
    beta=0.1,
    # -- Misc --
    logging_steps=5,
    save_steps=200,
    report_to="none",
    remove_unused_columns=False,
    seed=SEED,
)

# Pass base_model (not policy_model) — DPOTrainer wraps it with LoRA internally via peft_config
dpo_trainer = DPOTrainer(
    model=base_model,
    ref_model=None,          # LoRA: frozen base is its own reference — no copy
    args=dpo_config,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
    peft_config=lora_config, # DPOTrainer applies LoRA adapters here
)

trainable = sum(p.numel() for p in dpo_trainer.model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in dpo_trainer.model.parameters())
print(f"Trainable params : {trainable/1e6:.2f}M / {total/1e6:.1f}M ({100*trainable/total:.2f}%)")
print()
print("Starting DPO training...")
train_result = dpo_trainer.train()
print(f"\nTraining complete! Final loss: {train_result.training_loss:.4f}")

#### Step 4: Training Loss Curve

In [ ]:
import matplotlib.pyplot as plt

# Extract training loss from trainer log history
log_history = dpo_trainer.state.log_history
train_logs  = [entry for entry in log_history if "loss" in entry]

steps  = [entry["step"] for entry in train_logs]
losses = [entry["loss"] for entry in train_logs]

plt.figure(figsize=(10, 5))
plt.plot(steps, losses, marker="o", linewidth=2, color="steelblue", label="Training Loss")
plt.xlabel("Training Steps", fontsize=12)
plt.ylabel("Loss",           fontsize=12)
plt.title("DPO Training Loss Curve — Qwen2.5-1.5B-Instruct", fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("dpo_loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Initial loss : {losses[0]:.4f}")
print(f"Final loss   : {losses[-1]:.4f}")
print(f"Reduction    : {losses[0] - losses[-1]:.4f}")

## Task 3. Pushing the Model to Hugging Face Hub (0.5 point)

1. Save the trained model and upload it to the Hugging Face Model Hub.
2. Provide a link to your uploaded model in the documentation.

In [ ]:
from huggingface_hub import whoami
info = whoami()
print(info["name"])


In [ ]:
from huggingface_hub import login
login()

In [ ]:
from huggingface_hub import login

# Silent login — reads token from .env (loaded earlier), no popup
login()
print(f"Logged in to Hugging Face")

HF_USERNAME = "sushmee"
MODEL_REPO  = f"{HF_USERNAME}/qwen2.5-1.5b-truthy-dpo"
LOCAL_SAVE  = "../model/dpo_model_final"

# Merge LoRA adapters into the base model weights before saving
print("Merging LoRA adapters into base model...")
merged_model = dpo_trainer.model.merge_and_unload()

# Save merged model & tokenizer locally
print(f"Saving merged model to: {LOCAL_SAVE}")
merged_model.save_pretrained(LOCAL_SAVE)
tokenizer.save_pretrained(LOCAL_SAVE)

# Push to Hub (token already set via login above)
print(f"\nPushing to Hugging Face Hub: {MODEL_REPO}")
merged_model.push_to_hub(MODEL_REPO)
tokenizer.push_to_hub(MODEL_REPO)

print(f"\nModel successfully uploaded!")
print(f"Model URL: https://huggingface.co/{MODEL_REPO}")

## Task 4. Evaluation: LLM-as-a-Judge with AlpacaEval (2 points)

Build a pipeline where a strong LLM (e.g., GPT-4o-mini, Gemini) acts as an automatic judge to compare the base model and the DPO-fine-tuned model side-by-side on 15 samples from AlpacaEval.

#### Step 1: Load AlpacaEval & Generate Responses

In [ ]:
import requests, json
from datasets import load_dataset

ALPACA_URL = (
    "https://huggingface.co/datasets/tatsu-lab/alpaca_eval"
    "/resolve/main/data/alpaca_eval.json"
)

alpaca_data = None

# Try direct HTTP download first
try:
    response = requests.get(ALPACA_URL, headers={"Authorization": f"Bearer {HF_TOKEN}"}, timeout=30)
    response.raise_for_status()
    alpaca_data = response.json()
    print(f"Loaded via URL ({len(alpaca_data)} examples)")
except Exception as e:
    print(f"URL fetch failed: {e}")

# Fall back to datasets library if URL failed
if alpaca_data is None:
    try:
        print("Trying load_dataset...")
        ds_alpaca  = load_dataset("tatsu-lab/alpaca_eval", token=HF_TOKEN, trust_remote_code=True)
        split_name = "eval" if "eval" in ds_alpaca else list(ds_alpaca.keys())[0]
        alpaca_data = [dict(x) for x in ds_alpaca[split_name]]
        print(f"Loaded via datasets library ({len(alpaca_data)} examples, split='{split_name}')")
    except Exception as e:
        print(f"datasets library also failed: {e}")
        raise

# Filter for the 'helpful_base' subset
helpful_base = [x for x in alpaca_data if x.get("dataset") == "helpful_base"]

# If no 'helpful_base' tag, just use first 15 examples
if not helpful_base:
    print("No 'helpful_base' subset found — using first 15 examples directly")
    test_samples = alpaca_data[:15]
else:
    test_samples = helpful_base[:15]

print(f"Total AlpacaEval examples  : {len(alpaca_data)}")
print(f"Selected for evaluation    : {len(test_samples)}")
print(f"\nSample instruction: {test_samples[0]['instruction'][:150]}...")

In [ ]:
LOCAL_SAVE = "../model/dpo_model_final"  # redefine in case Task 3 cell wasn't run

def generate_response(mdl, tok, instruction, max_new_tokens=256):
    """Generate a response from a causal LM given a plain-text instruction."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": instruction},
    ]
    text   = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(mdl.device)   # move inputs to same device as model

    with torch.no_grad():
        output_ids = mdl.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tok.decode(new_tokens, skip_special_tokens=True)


# ── Base model responses ──────────────────────────────────────────────────────
print("Loading base model for comparison...")
base_model_eval = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, token=HF_TOKEN
).to(device)
base_model_eval.eval()

print("Generating base model responses...")
base_answers = []
for sample in tqdm(test_samples, desc="Base model"):
    base_answers.append(generate_response(base_model_eval, tokenizer, sample["instruction"]))

# Free memory before loading DPO model
del base_model_eval
torch.mps.empty_cache()

# ── DPO model responses ───────────────────────────────────────────────────────
print("\nLoading DPO fine-tuned model...")
dpo_model_eval = AutoModelForCausalLM.from_pretrained(
    LOCAL_SAVE, torch_dtype=torch.float16
).to(device)
dpo_model_eval.eval()

print("Generating DPO model responses...")
eval_results = []
for i, (sample, base_ans) in enumerate(tqdm(zip(test_samples, base_answers), desc="DPO model", total=len(test_samples))):
    dpo_ans = generate_response(dpo_model_eval, tokenizer, sample["instruction"])
    eval_results.append({
        "sample_id":   i + 1,
        "instruction": sample["instruction"],
        "base_answer": base_ans,
        "dpo_answer":  dpo_ans,
    })

del dpo_model_eval
torch.mps.empty_cache()

print(f"\nGeneration complete! ({len(eval_results)} samples)")

#### Step 2: The Judge Prompt

In [ ]:
JUDGE_PROMPT_TEMPLATE = """\
You are a highly qualified and impartial judge evaluating two AI models. \
Your task is to determine which model provides a better, more accurate, \
and more helpful response to the user's instruction.

User Instruction: {instruction}

Model A (Base Model): {base_answer}

Model B (DPO Model): {dpo_answer}

Evaluate both models. Output ONLY your final verdict as exactly one of the \
following options, with no extra text or explanation: "Model A", "Model B", or "Tie".\
"""

print("Judge Prompt Template (first 600 chars):")
print(JUDGE_PROMPT_TEMPLATE[:600])

#### Step 3: Evaluate and Collect Results

In [ ]:
import anthropic

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
judge_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def call_judge(instruction, base_answer, dpo_answer):
    """Call the LLM judge and return a normalised verdict."""
    prompt = JUDGE_PROMPT_TEMPLATE.format(
        instruction=instruction,
        base_answer=base_answer,
        dpo_answer=dpo_answer,
    )
    message = judge_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=16,
        messages=[{"role": "user", "content": prompt}],
    )
    raw = message.content[0].text.strip()
    if   "Model B" in raw: return "Model B"
    elif "Model A" in raw: return "Model A"
    elif "Tie"     in raw: return "Tie"
    else:                  return "Unknown"

print("Running LLM-as-a-Judge evaluation...")
for result in tqdm(eval_results, desc="Judging"):
    result["winner"] = call_judge(
        result["instruction"],
        result["base_answer"],
        result["dpo_answer"],
    )

print("Evaluation complete!")

In [ ]:
import pandas as pd

results_df = pd.DataFrame([
    {
        "Sample ID":             r["sample_id"],
        "Instruction (Truncated)": r["instruction"][:65] + "...",
        "Winner (Judge)":        r["winner"],
    }
    for r in eval_results
])

print(results_df.to_string(index=False))

#### Step 4: Calculate Win Rate

In [ ]:
total        = len(eval_results)
model_b_wins = sum(1 for r in eval_results if r["winner"] == "Model B")
model_a_wins = sum(1 for r in eval_results if r["winner"] == "Model A")
ties         = sum(1 for r in eval_results if r["winner"] == "Tie")
unknown      = sum(1 for r in eval_results if r["winner"] == "Unknown")
valid        = total - unknown

# Win Rate formula from the assignment
win_rate = (model_b_wins + 0.5 * ties) / valid * 100 if valid > 0 else 0.0

print("=" * 52)
print("  EVALUATION RESULTS SUMMARY")
print("=" * 52)
print(f"  Total samples evaluated : {total}")
print(f"  Valid evaluations       : {valid}")
print(f"  Model B (DPO) Wins      : {model_b_wins}")
print(f"  Model A (Base) Wins     : {model_a_wins}")
print(f"  Ties                    : {ties}")
print(f"  Unknown / Invalid       : {unknown}")
print("=" * 52)
print(f"  Win Rate (DPO Model)    : {win_rate:.2f}%")
print("=" * 52)

if win_rate > 50:
    conclusion = "DPO training successfully improved the model — it outperforms the base model on the AlpacaEval benchmark."
elif win_rate == 50:
    conclusion = "DPO training resulted in a roughly equal performance between the base and DPO model."
else:
    conclusion = "DPO training did not clearly improve the model on this benchmark; further hyperparameter tuning may help."

print(f"\nConclusion: {conclusion}")